# Continuous Assessment - 2 (CA2): Market Basket Analysis & Product Recommendation

**Course:** Data Mining and Business Intelligence (DMBI), V.E.S.I.T., Sem-5, D15B  
**Team:** Vedant Gharat (50), Ayush Lokhande (55), Deepak Nishad (59), Ayush Yadav (69)  
**Guide:** Sukanya Roychowdhury  
**Dataset:** UCI Machine Learning Repository – Online Retail (Dataset ID 352)  

---

## 1. Problem Statement & Research Goal
> *How can historical retail transactions be analysed to discover which products are frequently purchased together, so that the retailer can generate data-driven product recommendations and cross-selling strategies?*


In [1]:
import sys
from pathlib import Path

# Setup sys.path for backend service imports
BACKEND_DIR = Path.cwd() / 'backend' if (Path.cwd() / 'backend').exists() else Path.cwd().parent / 'backend'
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from app.services.preprocessing import load_raw_dataframe, generate_raw_profile, run_preprocessing_pipeline
from app.services.eda import run_eda_analysis
from app.services.mining import execute_mining_run
from app.services.recommender import get_default_recommender, run_offline_evaluation, build_rule_decision_matrix
print('Environment and services successfully imported.')

Environment and services successfully imported.


## 2. Raw Data Inspection & Profile Verification
Loading raw dataset records and computing the empirical dataset profile directly from the file.

In [2]:
raw_df = load_raw_dataframe()
print('Raw Dataset Dimensions:', raw_df.shape)
raw_profile = generate_raw_profile(raw_df)
print('Total Records:', raw_profile['computed']['total_records'])
print('Distinct Invoices:', raw_profile['computed']['distinct_invoices'])
print('Distinct Products:', raw_profile['computed']['distinct_products'])
print('Distinct Customers:', raw_profile['computed']['distinct_customers'])

Loading cached raw dataset from D:\DMBI CA\backend\data\processed\raw.parquet...
Raw Dataset Dimensions: (541909, 8)
Total Records: 541909
Distinct Invoices: 25900
Distinct Products: 4070
Distinct Customers: 4372


## 3. 11-Step Data Preprocessing Pipeline
Executing the reproducible cleaning pipeline with row removal logging.

In [3]:
baskets_df, matrix, label_map, log = run_preprocessing_pipeline()
log_df = pd.DataFrame(log)
print(log_df[['step', 'rule', 'rows_before', 'rows_after', 'rows_removed']])

Loading cached raw dataset from D:\DMBI CA\backend\data\processed\raw.parquet...
Step 02 [Missing keys filter]: 541909 -> 541909 (removed 0)
Step 03 [Cancellation filter]: 541909 -> 532621 (removed 9288)
Step 04 [Invalid values filter]: 532621 -> 530104 (removed 2517)
Step 05 [Non-product entries filter]: 530104 -> 527725 (removed 2379)
Step 06 [Exact duplicates filter]: 527725 -> 522504 (removed 5221)
Step 07 [Text cleaning & Canonical labels]: Mapped 3900 distinct products
Step 08 [United Kingdom scope filter]: 522504 -> 478838 (removed 43666)
Step 09 [Basket grouping]: Created 17901 unique transaction baskets
Step 10 [Basket-size filter]: 17901 -> 16506 (removed 1395 single-item baskets)
Step 11 [One-hot encode]: Filtered 3890 items down to 1628 frequent items at 0.5% floor

PREPROCESSING COMPLETE:
  Valid UK Invoices (>=2 items): 16506
  Products meeting support floor: 1628
  Average basket size: 28.61
  Boolean Matrix shape: (16429, 1628)

    step                              rul

D:\DMBI CA\backend\app\services\preprocessing.py:177: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  curr_df["StockCode"] = curr_df["StockCode"].astype(str).str.strip()
D:\DMBI CA\backend\app\services\preprocessing.py:205: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  curr_df["Description"] = (
D:\DMBI CA\backend\app\services\preprocessing.py:241: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveat

## 4. Exploratory Data Analysis (EDA)
Computing retail sales patterns, seasonality, and CA1 Visualisation #1.

In [4]:
eda_summary = run_eda_analysis()
top_products = pd.DataFrame(eda_summary['top_products'])
print('Top 10 Products by Valid UK Invoices:')
print(top_products[['rank', 'stock_code', 'label', 'invoice_count', 'share_of_baskets_pct']])

EDA Analysis complete: saved to D:\DMBI CA\backend\results\eda_summary.json
Top 10 Products by Valid UK Invoices:
   rank stock_code  ... invoice_count  share_of_baskets_pct
0     1     85123A  ...          2086                 12.64
1     2     85099B  ...          1920                 11.63
2     3      22423  ...          1665                 10.09
3     4      47566  ...          1578                  9.56
4     5      20725  ...          1392                  8.43
5     6      84879  ...          1369                  8.29
6     7      22197  ...          1273                  7.71
7     8      22720  ...          1233                  7.47
8     9      22457  ...          1218                  7.38
9    10      20727  ...          1214                  7.35

[10 rows x 5 columns]


## 5. Association Rule Mining (FP-Growth)
Mining frequent itemsets and generating association rules evaluated by Support, Confidence, and Lift.

In [5]:
result = execute_mining_run(algorithm='fpgrowth', min_support=0.01, min_confidence=0.50, min_lift=1.0)
print(f'Mined {result["n_itemsets"]} frequent itemsets and {result["n_rules"]} rules in {result["runtime_seconds"]:.2f}s.')

# Display Top 5 Rules by Lift
for rank, r in enumerate(result['rules'][:5], 1):
    ant = ' + '.join([a['label'] for a in r['antecedents']])
    con = ' + '.join([c['label'] for c in r['consequents']])
    print(f'{rank}. [{ant}] -> [{con}] | Lift: {r["lift"]:.2f}x | Conf: {r["confidence"]*100:.1f}% | Supp: {r["support"]*100:.2f}%')

Mined 2830 frequent itemsets and 1865 rules in 47.29s.
1. [HERB MARKER ROSEMARY + HERB MARKER CHIVES] -> [HERB MARKER THYME + HERB MARKER PARSLEY] | Lift: 75.03x | Conf: 94.5% | Supp: 1.05%
2. [HERB MARKER THYME + HERB MARKER PARSLEY] -> [HERB MARKER ROSEMARY + HERB MARKER CHIVES] | Lift: 75.03x | Conf: 83.6% | Supp: 1.05%
3. [HERB MARKER ROSEMARY + HERB MARKER PARSLEY] -> [HERB MARKER THYME + HERB MARKER CHIVES] | Lift: 74.62x | Conf: 83.6% | Supp: 1.05%
4. [HERB MARKER THYME + HERB MARKER CHIVES] -> [HERB MARKER ROSEMARY + HERB MARKER PARSLEY] | Lift: 74.62x | Conf: 94.0% | Supp: 1.05%
5. [HERB MARKER ROSEMARY + HERB MARKER CHIVES] -> [HERB MARKER THYME + HERB MARKER MINT] | Lift: 74.51x | Conf: 90.7% | Supp: 1.01%


## 6. Product Recommender & Cart Simulator
Testing real-time single-item and multi-item basket recommendations with explanations.

In [6]:
recommender = get_default_recommender()
single_rec = recommender.recommend_for_product('85099B', k=3)
print('Single Product Recommendation for:', single_rec['query_product'])
for rec in single_rec['recommendations']:
    print(f' - [{rec["code"]}] {rec["label"]} (Lift: {rec["lift"]:.1f}x, Conf: {rec["confidence"]*100:.0f}%)')

cart_rec = recommender.recommend_for_cart(['85099B', '22386'], k=3)
print('\nCart Recommendations for Multi-Item Basket:')
for rec in cart_rec['recommendations']:
    print(f' - [{rec["code"]}] {rec["label"]} | {rec["explanation"]}')

Single Product Recommendation for: {'code': '85099B', 'label': 'JUMBO BAG RED RETROSPOT'}
 - [20712] JUMBO BAG WOODLAND ANIMALS (Lift: 21.4x, Conf: 52%)
 - [22386] JUMBO BAG PINK POLKADOT (Lift: 21.4x, Conf: 52%)
 - [21931] JUMBO STORAGE BAG SUKI (Lift: 20.1x, Conf: 52%)

Cart Recommendations for Multi-Item Basket:
 - [21931] JUMBO STORAGE BAG SUKI | Recommended because your cart contains [JUMBO BAG PINK POLKADOT + JUMBO BAG RED RETROSPOT] (Confidence: 51.6%, Lift: 7.5x).


## 7. Offline Evaluation & Rule-to-Decision Matrix
Evaluating generalisability on unseen future transactions (80/20 train/test split) and mapping rules to commercial decisions.

In [7]:
eval_data = run_offline_evaluation()
eval_df = pd.DataFrame(eval_data['metrics'])
print('Offline Evaluation Results (Rule Recommender vs Popularity Baseline):')
print(eval_df)

decision_matrix = build_rule_decision_matrix()
print('\nRule-to-Decision Matrix Sample:')
for row in decision_matrix:
    print(f'[{row["category"]}] {row["possible_decision"]} (Based on: {row["mining_insight"][:40]}...)')

Running Recommender Offline Evaluation (80/20 Time-Based Split)...
Split: 13204 train baskets, 3302 test baskets
Mined 1889 rules on train set in 44.72s
Offline Evaluation saved to D:\DMBI CA\backend\results\recommender_eval.json
Offline Evaluation Results (Rule Recommender vs Popularity Baseline):
   k  rule_hit_rate_pct  popularity_hit_rate_pct  difference_pct
0  1               4.63                     0.55            4.08
1  3               7.15                     1.30            5.85
2  5               8.03                     1.91            6.12
Decision Matrix saved to D:\DMBI CA\backend\results\decision_matrix.json

Rule-to-Decision Matrix Sample:
[Cross-Selling / Recommendation] Test as an on-page 'Frequently Bought Together' widget and cart add-on. (Based on: Pair [REGENCY SUGAR BOWL GREEN] -> [REGE...)
[Product Bundling] Package products together as a discounted multi-pack or gift set bundle. (Based on: Set [HERB MARKER ROSEMARY] -> [HERB MARK...)
[Popularity Guardrail] Do